# PDF Taxonomy Tagger

This notebook processes PDF files paragraph-by-paragraph and matches each paragraph to a hierarchical taxonomy using LLM-based scoring.

## Features
- Extract paragraphs from PDFs
- Hierarchical matching with confidence scores
- Partial path support (stops at confident levels)
- Tracks new branches for quality assessment
- Batch processing support
- Colab-ready with multiple API key options


## 1. Installation & Setup


In [ ]:
# Install required packages
!pip install pdfplumber openai anthropic python-dotenv tqdm -q


## 2. API Key Configuration


In [ ]:
import os
import json
import sys
from pathlib import Path

# Check if running in Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Running in Google Colab")
    from google.colab import drive, userdata
    
    # Option 1: Try to load from Colab secrets/userdata
    try:
        API_KEY = userdata.get('OPENAI_API_KEY') or userdata.get('ANTHROPIC_API_KEY')
        DEFAULT_MODEL = userdata.get('DEFAULT_MODEL', 'gpt-4')
        print("✓ Loaded API key from Colab secrets")
    except:
        # Option 2: Mount Drive and try to load secrets.json
        try:
            drive.mount('/content/drive')
            secrets_path = '/content/drive/MyDrive/secrets.json'
            if os.path.exists(secrets_path):
                with open(secrets_path) as f:
                    secrets = json.load(f)
                    API_KEY = secrets.get('openai_api_key') or secrets.get('anthropic_api_key')
                    DEFAULT_MODEL = secrets.get('default_model', 'gpt-4')
                print("✓ Loaded API key from Drive secrets.json")
            else:
                raise FileNotFoundError
        except:
            # Option 3: Direct input
            print("\n⚠️ No API key found. Please enter it manually:")
            API_KEY = input("Enter your API key: ")
            DEFAULT_MODEL = input("Enter model name (default: gpt-4): ") or 'gpt-4'
else:
    print("Running locally")
    # Try to load from secrets.json
    secrets_file = Path('secrets.json')
    if secrets_file.exists():
        with open(secrets_file) as f:
            secrets = json.load(f)
            API_KEY = secrets.get('openai_api_key') or secrets.get('anthropic_api_key')
            DEFAULT_MODEL = secrets.get('default_model', 'gpt-4')
        print("✓ Loaded API key from secrets.json")
    else:
        print("\n⚠️ secrets.json not found. Copy secrets_template.json to secrets.json and add your API key.")
        API_KEY = input("Enter your API key: ")
        DEFAULT_MODEL = input("Enter model name (default: gpt-4): ") or 'gpt-4'

# Set environment variable
if 'gpt' in DEFAULT_MODEL.lower():
    os.environ['OPENAI_API_KEY'] = API_KEY
    print(f"✓ Using OpenAI model: {DEFAULT_MODEL}")
else:
    os.environ['ANTHROPIC_API_KEY'] = API_KEY
    print(f"✓ Using Anthropic model: {DEFAULT_MODEL}")


## 3. Load Taxonomy


In [ ]:
# Load taxonomy
taxonomy_file = Path('taxonomy_gs3.json')

if not taxonomy_file.exists():
    print("⚠️ taxonomy_gs3.json not found in current directory")
    if IN_COLAB:
        print("Please upload it or clone the GitHub repo")
else:
    with open(taxonomy_file, 'r', encoding='utf-8') as f:
        TAXONOMY = json.load(f)
    
    print(f"✓ Loaded taxonomy with {len(TAXONOMY)} top-level topics:")
    for topic in TAXONOMY.keys():
        print(f"  - {topic}")


## 4. Configuration & Prompt Templates


In [ ]:
# Matching configuration - Easy to modify
MATCHING_CONFIG = {
    "threshold": 70,              # Minimum score to accept a match
    "min_score_diff": 10,         # Minimum gap between top match and second
    "llm_model": DEFAULT_MODEL,
    "temperature": 0.3,           # Lower = more deterministic
    "max_retries": 3,
    "debug_mode": True            # Print detailed logs
}

# Prompt templates - Easy to experiment with
PROMPTS = {
    "system": "You are an expert at categorizing educational content into hierarchical taxonomies. Your task is to analyze text and determine how well it matches given topics.",
    
    "matching": """Analyze this paragraph and rate how well it matches each of the following topics.

Paragraph:
{paragraph}

Topics to match against:
{options}

Rate each topic on a scale of 0-100 where:
- 0-30: Not related
- 31-60: Somewhat related
- 61-80: Closely related
- 81-100: Perfect match

Return ONLY a valid JSON object with topic names as keys and integer scores as values.
Example: {{"Topic A": 85, "Topic B": 45, "Topic C": 20}}""",

    "matching_short": """Rate how well this paragraph matches each topic (0-100):

Paragraph: {paragraph}

Topics: {options}

Return JSON only: {{"Topic": score, ...}}"""
}

print("✓ Configuration loaded")
print(f"  Threshold: {MATCHING_CONFIG['threshold']}")
print(f"  Model: {MATCHING_CONFIG['llm_model']}")
print(f"  Debug mode: {MATCHING_CONFIG['debug_mode']}")


## 5. Import Libraries


In [ ]:
import pdfplumber
import re
from datetime import datetime
from typing import List, Dict, Tuple, Optional
from tqdm.auto import tqdm
import time

# Import LLM clients
if 'gpt' in MATCHING_CONFIG['llm_model'].lower():
    from openai import OpenAI
    llm_client = OpenAI(api_key=API_KEY)
    LLM_PROVIDER = 'openai'
else:
    from anthropic import Anthropic
    llm_client = Anthropic(api_key=API_KEY)
    LLM_PROVIDER = 'anthropic'

print(f"✓ Libraries imported, using {LLM_PROVIDER}")


## 6. PDF Text Extraction


In [ ]:
def extract_paragraphs_from_pdf(pdf_path: str) -> List[Dict]:
    """
    Extract paragraphs from PDF with page numbers.
    
    Args:
        pdf_path: Path to PDF file
        
    Returns:
        List of dicts with 'text', 'page_number', 'paragraph_number'
    """
    paragraphs = []
    paragraph_counter = 0
    
    print(f"\nExtracting text from: {pdf_path}")
    
    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, 1):
            text = page.extract_text()
            
            if not text:
                continue
            
            # Split by double newlines to identify paragraphs
            raw_paragraphs = re.split(r'\n\s*\n', text)
            
            for para_text in raw_paragraphs:
                # Clean up the paragraph
                para_text = para_text.strip()
                para_text = re.sub(r'\s+', ' ', para_text)  # Normalize whitespace
                
                # Skip very short paragraphs (likely headers or page numbers)
                if len(para_text) < 30:
                    continue
                
                paragraph_counter += 1
                paragraphs.append({
                    'text': para_text,
                    'page_number': page_num,
                    'paragraph_number': paragraph_counter
                })
    
    print(f"✓ Extracted {len(paragraphs)} paragraphs from {len(pdf.pages)} pages")
    return paragraphs

print("✓ PDF extraction function defined")


## 7. LLM Matching System (Modular)


In [ ]:
def build_prompt(paragraph: str, options: List[str], level: str) -> str:
    """Build LLM prompt for matching."""
    # Truncate paragraph if too long (to save tokens)
    max_para_length = 500
    if len(paragraph) > max_para_length:
        paragraph = paragraph[:max_para_length] + "..."
    
    # Format options as numbered list
    options_text = "\n".join([f"{i+1}. {opt}" for i, opt in enumerate(options)])
    
    # Use shorter prompt for subtopics to save tokens
    if level != 'topic':
        template = PROMPTS['matching_short']
    else:
        template = PROMPTS['matching']
    
    return template.format(paragraph=paragraph, options=options_text)


def call_llm(prompt: str, config: Dict) -> str:
    """Call LLM API with retry logic."""
    for attempt in range(config['max_retries']):
        try:
            if LLM_PROVIDER == 'openai':
                response = llm_client.chat.completions.create(
                    model=config['llm_model'],
                    messages=[
                        {"role": "system", "content": PROMPTS['system']},
                        {"role": "user", "content": prompt}
                    ],
                    temperature=config['temperature'],
                    max_tokens=300
                )
                return response.choices[0].message.content
            else:  # anthropic
                response = llm_client.messages.create(
                    model=config['llm_model'],
                    max_tokens=300,
                    temperature=config['temperature'],
                    system=PROMPTS['system'],
                    messages=[{"role": "user", "content": prompt}]
                )
                return response.content[0].text
        except Exception as e:
            if attempt < config['max_retries'] - 1:
                print(f"  ⚠️ API error, retrying... ({e})")
                time.sleep(2 ** attempt)  # Exponential backoff
            else:
                print(f"  ❌ API error after {config['max_retries']} attempts: {e}")
                raise


def parse_scores(response: str) -> Dict[str, int]:
    """Parse LLM response to extract scores."""
    try:
        # Try to find JSON in the response
        json_match = re.search(r'\{[^}]+\}', response, re.DOTALL)
        if json_match:
            scores = json.loads(json_match.group())
            # Convert to int if needed
            return {k: int(v) for k, v in scores.items()}
        else:
            print(f"  ⚠️ No JSON found in response: {response[:100]}")
            return {}
    except Exception as e:
        print(f"  ⚠️ Error parsing scores: {e}")
        print(f"  Response was: {response[:200]}")
        return {}


def evaluate_match(scores: Dict[str, int], threshold: int, min_diff: int) -> Tuple[Optional[str], int]:
    """Evaluate if we have a confident match."""
    if not scores:
        return None, 0
    
    # Sort by score descending
    sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    top_match, top_score = sorted_scores[0]
    
    # Check if top score meets threshold
    if top_score < threshold:
        return None, top_score
    
    # Check if there's a clear winner (optional)
    if len(sorted_scores) > 1 and min_diff > 0:
        second_score = sorted_scores[1][1]
        if top_score - second_score < min_diff:
            return None, top_score
    
    return top_match, top_score

print("✓ Helper functions defined")


In [ ]:
def match_at_level(paragraph: str, options: List[str], level: str, config: Dict) -> Tuple[Optional[str], int, Dict]:
    """Match paragraph against options at a specific taxonomy level."""
    if not options:
        return None, 0, {}
    
    if config['debug_mode']:
        print(f"  Matching at {level} level: {len(options)} options")
    
    # Build and send prompt
    prompt = build_prompt(paragraph, options, level)
    
    if config['debug_mode']:
        print(f"  Prompt length: {len(prompt)} chars")
    
    response = call_llm(prompt, config)
    
    if config['debug_mode']:
        print(f"  LLM response: {response[:150]}...")
    
    # Parse scores
    scores = parse_scores(response)
    
    # Evaluate match
    matched, score = evaluate_match(
        scores, 
        config['threshold'], 
        config['min_score_diff']
    )
    
    if config['debug_mode']:
        if matched:
            print(f"  ✓ Matched: {matched} (score: {score})")
        else:
            print(f"  ✗ No confident match (top score: {score})")
    
    return matched, score, scores


def match_hierarchically(paragraph: str, taxonomy: Dict, config: Dict) -> Dict:
    """Match paragraph hierarchically through taxonomy."""
    result = {
        'taxonomy_path': [],
        'match_scores': [],
        'is_new_branch': False,
        'match_depth': 0
    }
    
    current_level = taxonomy
    level_names = ['topic', 'subtopic', 'sub-subtopic', 'leaf']
    level_idx = 0
    
    if config['debug_mode']:
        print(f"\n--- Matching paragraph (first 100 chars): {paragraph[:100]}...")
    
    while current_level and isinstance(current_level, dict) and len(current_level) > 0:
        # Get options at current level
        options = list(current_level.keys())
        level_name = level_names[level_idx] if level_idx < len(level_names) else f'level_{level_idx}'
        
        # Try to match
        matched, score, all_scores = match_at_level(paragraph, options, level_name, config)
        
        # Record the attempt
        result['match_scores'].append({
            'level': level_name,
            'matched': matched,
            'score': score,
            'all_scores': all_scores
        })
        
        # Check if match was successful
        if matched is None:
            # No confident match - mark as new branch and stop
            result['is_new_branch'] = True
            if config['debug_mode']:
                print(f"  Stopping at depth {len(result['taxonomy_path'])}")
            break
        
        # Add to path and drill down
        result['taxonomy_path'].append(matched)
        result['match_depth'] += 1
        current_level = current_level[matched]
        level_idx += 1
        
        # Check if we've reached a leaf (empty dict)
        if not current_level or not isinstance(current_level, dict) or len(current_level) == 0:
            if config['debug_mode']:
                print(f"  ✓ Reached leaf node at depth {result['match_depth']}")
            break
    
    return result

print("✓ LLM matching functions defined")


## 8. Output Functions


In [ ]:
def create_paragraph_record(paragraph_data: Dict, match_result: Dict, pdf_name: str) -> Dict:
    """Create a complete record for a tagged paragraph."""
    return {
        'paragraph_id': f"{pdf_name}_{paragraph_data['paragraph_number']:03d}",
        'text': paragraph_data['text'],
        'taxonomy_path': match_result['taxonomy_path'],
        'match_depth': match_result['match_depth'],
        'match_scores': match_result['match_scores'],
        'is_new_branch': match_result['is_new_branch'],
        'metadata': {
            'pdf_file': pdf_name,
            'page_number': paragraph_data['page_number'],
            'paragraph_number': paragraph_data['paragraph_number'],
            'timestamp': datetime.now().isoformat(),
            'processing_config': {
                'threshold': MATCHING_CONFIG['threshold'],
                'model': MATCHING_CONFIG['llm_model']
            }
        }
    }


def save_results(records: List[Dict], output_path: str):
    """Save tagged paragraphs to JSON file."""
    # Create output directory if needed
    output_dir = Path(output_path).parent
    output_dir.mkdir(exist_ok=True, parents=True)
    
    # Save to file
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(records, f, indent=2, ensure_ascii=False)
    
    print(f"\n✓ Saved {len(records)} records to {output_path}")
    
    # Print summary statistics
    new_branches = sum(1 for r in records if r['is_new_branch'])
    fully_matched = sum(1 for r in records if not r['is_new_branch'])
    
    print(f"\nSummary:")
    print(f"  Fully matched: {fully_matched} ({fully_matched/len(records)*100:.1f}%)")
    print(f"  New branches: {new_branches} ({new_branches/len(records)*100:.1f}%)")
    
    # Show depth distribution
    depths = {}
    for r in records:
        depth = r['match_depth']
        depths[depth] = depths.get(depth, 0) + 1
    
    print(f"\nMatch depth distribution:")
    for depth in sorted(depths.keys()):
        print(f"  Depth {depth}: {depths[depth]} paragraphs")

print("✓ Output functions defined")


## 9. Batch Processing & Aggregation


In [ ]:
def process_pdf(pdf_path: str, config: Dict, taxonomy: Dict) -> List[Dict]:
    """Process a single PDF file."""
    pdf_name = Path(pdf_path).stem
    
    # Extract paragraphs
    paragraphs = extract_paragraphs_from_pdf(pdf_path)
    
    # Process each paragraph
    records = []
    print(f"\nProcessing {len(paragraphs)} paragraphs...")
    
    for para in tqdm(paragraphs, desc="Matching paragraphs"):
        try:
            match_result = match_hierarchically(para['text'], taxonomy, config)
            record = create_paragraph_record(para, match_result, pdf_name)
            records.append(record)
        except Exception as e:
            print(f"\n⚠️ Error processing paragraph {para['paragraph_number']}: {e}")
            continue
    
    return records


def process_multiple_pdfs(pdf_paths: List[str], config: Dict, taxonomy: Dict) -> Dict[str, List[Dict]]:
    """Process multiple PDFs."""
    all_results = {}
    
    for pdf_path in pdf_paths:
        print(f"\n{'='*60}")
        print(f"Processing: {pdf_path}")
        print(f"{'='*60}")
        
        records = process_pdf(pdf_path, config, taxonomy)
        pdf_name = Path(pdf_path).stem
        all_results[pdf_name] = records
        
        # Save individual PDF results
        output_path = f"output/{pdf_name}_tagged.json"
        save_results(records, output_path)
    
    return all_results


def aggregate_by_topic(all_results: Dict[str, List[Dict]]) -> Dict[str, List[Dict]]:
    """Aggregate all paragraphs by top-level topic for vector DB preparation."""
    topic_data = {}
    
    for pdf_name, records in all_results.items():
        for record in records:
            if record['taxonomy_path']:  # Has at least a topic
                topic = record['taxonomy_path'][0]
                if topic not in topic_data:
                    topic_data[topic] = []
                topic_data[topic].append(record)
            else:  # No topic matched
                if 'UNMATCHED' not in topic_data:
                    topic_data['UNMATCHED'] = []
                topic_data['UNMATCHED'].append(record)
    
    # Save aggregated data
    output_dir = Path('output')
    output_dir.mkdir(exist_ok=True)
    
    for topic, records in topic_data.items():
        # Sanitize topic name for filename
        safe_topic = topic.replace(' ', '_').replace('&', 'and')
        output_path = output_dir / f"{safe_topic}_all_paragraphs.json"
        
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(records, f, indent=2, ensure_ascii=False)
        
        print(f"  {topic}: {len(records)} paragraphs -> {output_path}")
    
    return topic_data

print("✓ Batch processing functions defined")


## 10. Interactive Processing

Run this cell to process PDF(s) with interactive prompts.


In [ ]:
# Interactive mode
print("\n" + "="*60)
print("PDF TAXONOMY TAGGER - Interactive Mode")
print("="*60)

# Ask for PDF path(s)
pdf_input = input("\nEnter PDF path (or comma-separated paths for batch): ")
pdf_paths = [p.strip() for p in pdf_input.split(',')]

# Validate paths
valid_paths = []
for path in pdf_paths:
    if Path(path).exists():
        valid_paths.append(path)
        print(f"  ✓ Found: {path}")
    else:
        print(f"  ⚠️ Not found: {path}")

if not valid_paths:
    print("\n❌ No valid PDF files found. Please check paths.")
else:
    # Ask about config
    use_config = input("\nUse config file? (y/n, default: n): ").lower() == 'y'
    
    if use_config:
        config_path = input("Enter config file path (default: config_template.json): ") or "config_template.json"
        if Path(config_path).exists():
            with open(config_path) as f:
                user_config = json.load(f)
                # Merge with defaults
                MATCHING_CONFIG.update(user_config)
            print(f"  ✓ Loaded config from {config_path}")
    else:
        # Ask for key settings
        threshold_input = input(f"\nMatching threshold (0-100, default: {MATCHING_CONFIG['threshold']}): ")
        if threshold_input:
            MATCHING_CONFIG['threshold'] = int(threshold_input)
        
        debug_input = input(f"Debug mode? (y/n, default: {'y' if MATCHING_CONFIG['debug_mode'] else 'n'}): ")
        if debug_input:
            MATCHING_CONFIG['debug_mode'] = debug_input.lower() == 'y'
    
    print(f"\n✓ Configuration:")
    print(f"  Threshold: {MATCHING_CONFIG['threshold']}")
    print(f"  Model: {MATCHING_CONFIG['llm_model']}")
    print(f"  Debug: {MATCHING_CONFIG['debug_mode']}")
    
    # Process!
    proceed = input("\nProceed with processing? (y/n): ").lower()
    if proceed == 'y':
        if len(valid_paths) == 1:
            # Single PDF
            records = process_pdf(valid_paths[0], MATCHING_CONFIG, TAXONOMY)
            pdf_name = Path(valid_paths[0]).stem
            output_path = f"output/{pdf_name}_tagged.json"
            save_results(records, output_path)
        else:
            # Multiple PDFs
            all_results = process_multiple_pdfs(valid_paths, MATCHING_CONFIG, TAXONOMY)
            
            # Ask about aggregation
            aggregate = input("\nAggregate by topic for vector DB? (y/n): ").lower() == 'y'
            if aggregate:
                print("\nAggregating by topic...")
                topic_data = aggregate_by_topic(all_results)
                print(f"\n✓ Created {len(topic_data)} topic-level files")
        
        print("\n" + "="*60)
        print("✓ PROCESSING COMPLETE")
        print("="*60)
    else:
        print("\nCancelled.")


## 11. Quick Test (Optional)

Run this cell to test with a sample paragraph without processing a full PDF.


In [ ]:
# Test with a sample paragraph
test_paragraph = """Planning is an essential process for economic development in India. 
The Planning Commission was established in 1950 to formulate five-year plans for the country's 
economic and social development. These plans set targets for various sectors and allocate resources 
accordingly. The indicative planning approach allows for market forces to operate while providing 
overall direction to the economy."""

print("Testing with sample paragraph:\n")
print(test_paragraph)
print("\n" + "="*60)

# Set debug mode on for testing
test_config = MATCHING_CONFIG.copy()
test_config['debug_mode'] = True

# Match
result = match_hierarchically(test_paragraph, TAXONOMY, test_config)

print("\n" + "="*60)
print("TEST RESULT:")
print(f"  Path: {' > '.join(result['taxonomy_path'])}")
print(f"  Depth: {result['match_depth']}")
print(f"  New branch: {result['is_new_branch']}")


## 12. Utility: View Results

Load and inspect saved results.


In [ ]:
def view_results(json_path: str, num_samples: int = 5):
    """View saved results with summary."""
    with open(json_path, 'r', encoding='utf-8') as f:
        records = json.load(f)
    
    print(f"\nLoaded {len(records)} records from {json_path}\n")
    
    # Show samples
    print(f"Sample records (showing {num_samples}):\n")
    for i, record in enumerate(records[:num_samples]):
        print(f"[{i+1}] {record['paragraph_id']}")
        print(f"    Path: {' > '.join(record['taxonomy_path']) if record['taxonomy_path'] else '(no match)'}")
        print(f"    Text: {record['text'][:100]}...")
        print(f"    New branch: {record['is_new_branch']}")
        print()

# Example usage:
# view_results('output/your_pdf_tagged.json')


---

## Ready to Use!

### Quick Start:
1. Run all cells above to set everything up
2. Run the "Interactive Processing" cell (#10) to process your PDF
3. Use "View Results" cell (#12) to inspect the output

### Next Steps:
- Adjust `MATCHING_CONFIG` to tune matching behavior
- Modify `PROMPTS` to experiment with different LLM instructions
- Use topic-aggregated JSON files to create vector databases

### Output Location:
All results are saved in the `output/` directory.

### Configuration Files:
- `secrets.json` - Your API keys (copy from `secrets_template.json`)
- `config_template.json` - Processing configuration options
